# V020: training-only decaying-controller feasibility

Prioritized from the user-supplied first-place report S01, with overlap in second/fourth-place S02/S04. No external solution code or datasets are fetched. These equations are independently derived hypotheses.

For a sum-integral controller, assume error e=q-p, I_t=d I_(t-1)+e_t and u_t=Kp e_t+Ki I_t. Eliminating I gives u_t=d u_(t-1)-(Kp+Ki)p_t+d Kp p_(t-1)+(Kp+Ki)q_t-d Kp q_(t-1). Constant and linear-in-step reference families are tested separately. For time-weighted integration I_t=d I_(t-1)+dt e_t, use the additional dt*p_t term and enforce the shared physical coefficients. These are hypotheses, not a claim about the actual simulator.

Use exactly the576 original-A-training breaths from V015, raw float64 precision. Fit the first70% of unclipped inspiratory control rows; diagnose later rows. Compare OLS and fixed64-draw fitting-only RANSAC separately. No validation pressures, test data, GPU, unseen-pressure predictor or score. A positive gate would justify input-only reconstruction research, not submission. A negative gate would not disprove all controller matching. Current V019 GPU fits remain unchanged.


In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,time
import numpy as np
import pandas as pd
P=Path.cwd();O=P/'outputs/controller_arx_feasibility_v020';O.mkdir(parents=True,exist_ok=True);started=time.monotonic()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
planpath=P/'research/controller_arx_protocol_v020.json'
assert sha(planpath)=='3a78c577b9935c1be8bdf12b7db7d0ea0fba3b4c6f63f11c923a5bbb2fc58838'
plan=json.loads(planpath.read_text());assert not (O/'summary.json').exists(),'Preserve previous results'
assert sha(P/'data/train.csv')==plan['approved_train_sha256']
assert sha(P/plan['sample_source'])==plan['sample_source_sha256']
assert sha(P/plan['original_A_split_file'])==plan['A_split_file_sha256']
assert sha(P/plan['source_document'])==plan['source_document_sha256']
prior=json.loads((P/plan['sample_source']).read_text());sample_ids=np.asarray(prior['selected_A_fit_breath_ids'],np.int32)
with np.load(P/plan['original_A_split_file']) as z:fit_ids=z['A_fit'].copy();val_ids=z['A_validation'].copy()
assert len(sample_ids)==576 and len(np.unique(sample_ids))==576 and np.isin(sample_ids,fit_ids).all() and not np.isin(sample_ids,val_ids).any()
meta=pd.read_csv(P/'data/train.csv',usecols=['breath_id','R','C'],dtype=dict(breath_id='int32',R='int8',C='int8')).iloc[::80].reset_index(drop=True)
selected=np.flatnonzero(meta.breath_id.isin(sample_ids).to_numpy());selected_set=set(selected.tolist())
assert len(selected)==576 and (meta.iloc[selected].groupby(['R','C']).size()==64).all()
frame=pd.read_csv(P/'data/train.csv',usecols=['breath_id','R','C','time_step','u_in','u_out','pressure'],skiprows=lambda i:i>0 and (i-1)//80 not in selected_set,dtype=dict(breath_id='int32',R='int8',C='int8',time_step='float64',u_in='float64',u_out='int8',pressure='float64'))
assert len(frame)==576*80 and np.isin(frame.breath_id.to_numpy(),fit_ids).all() and not np.isin(frame.breath_id.to_numpy(),val_ids).any()

def physical_coefficients(coef,family,reference_steps):
    tol=plan['physical_coefficient_tolerance'];d=float(coef[0]);positive_d=1e-8<d<=1+tol
    if family.startswith('decay_sum'):
        a,b,c=map(float,coef[1:4]);kp=b/d if abs(d)>1e-12 else np.nan;ki=a-kp;den=a-b
        if den<=1e-8:return False,dict(decay=d,kp=kp,ki=ki,q0=np.nan,slope=np.nan)
        slope=float(coef[4])/den if family=='decay_sum_linear_step_reference' else 0.
        q0=(c-b*slope)/den;valid=positive_d and kp>1e-8 and ki>=-tol
    else:
        kp,b,ki,c,h=map(float,coef[1:6]);slope=0.
        if abs(ki)>1e-10:q0=h/ki
        elif abs(kp*(1-d))>1e-10:q0=c/(kp*(1-d))
        else:q0=np.nan
        near=lambda a,b:abs(a-b)<=tol*(1+abs(a)+abs(b))
        valid=positive_d and kp>1e-8 and ki>=-tol and near(b,d*kp) and np.isfinite(q0) and near(c,kp*(1-d)*q0) and near(h,ki*q0)
    q=q0+slope*reference_steps;valid=valid and np.isfinite(q).all() and q.min()>=-tol and q.max()<=100+tol
    return bool(valid),dict(decay=d,kp=kp,ki=ki,q0=q0,slope=slope)

def fit_linear(X,target,positions):
    Xf=X[positions];yf=target[positions];scale=np.maximum(np.sqrt(np.mean(Xf**2,axis=0)),1e-12);Z=Xf/scale
    if np.linalg.matrix_rank(Z)<Z.shape[1]:return None
    condition=float(np.linalg.cond(Z))
    if condition>=plan['maximum_condition']:return None
    coef=np.linalg.lstsq(Z,yf,rcond=None)[0]/scale
    return coef,condition

rows=[]
for bid,b in frame.groupby('breath_id',sort=False):
    u=b.u_in.to_numpy();p=b.pressure.to_numpy();t=b.time_step.to_numpy();phase=b.u_out.to_numpy()
    for family_i,family in enumerate(plan['families']):
        for lag in plan['pressure_lags']:
            k=np.arange(2,80);n=k-lag
            ok=(phase[k]==0)&(phase[k-1]==0)&(phase[n]==0)&(phase[n-1]==0)&(u[k]>1e-6)&(u[k]<99.999999)&(u[k-1]>1e-6)&(u[k-1]<99.999999)
            k=k[ok];n=k-lag
            if len(k)<plan['minimum_eligible_points']:continue
            dt=t[k]-t[k-1];X=np.column_stack([u[k-1],-p[n],p[n-1],np.ones(len(k))])
            if family=='decay_sum_linear_step_reference':X=np.column_stack([X,n.astype(float)])
            if family=='decay_dt_constant_reference':X=np.column_stack([u[k-1],-p[n],p[n-1],-dt*p[n],np.ones(len(k)),dt])
            target=u[k];cut=max(X.shape[1]+2,int(plan['chronological_fit_fraction']*len(k)))
            if len(k)-cut<plan['minimum_held_control_points']:continue
            reference_steps=np.arange(int(np.flatnonzero(phase==0).max())+1,dtype=float)
            ordinary=fit_linear(X,target,np.arange(cut))
            if ordinary is None:continue
            for estimator in plan['estimators']:
                candidates=[ordinary]
                if estimator=='training_only_RANSAC64':
                    rng=np.random.default_rng(plan['RANSAC_seed']+int(bid)*17+family_i*2+lag)
                    for _ in range(plan['RANSAC_draws']):
                        trial=fit_linear(X,target,np.sort(rng.choice(cut,size=X.shape[1],replace=False)))
                        if trial is not None:candidates.append(trial)
                evaluated=[]
                for coef,condition in candidates:
                    physical,params=physical_coefficients(coef,family,reference_steps)
                    fit_error=np.abs(X[:cut]@coef-target[:cut]);inliers=int((fit_error<1e-4).sum())
                    # The frozen fitting-control inlier-count/median rule alone
                    # selects a candidate; physical checks gate it afterward.
                    key=(inliers,-float(np.median(fit_error)))
                    evaluated.append((key,coef,condition,physical,params))
                _,coef,condition,physical,params=max(evaluated,key=lambda a:a[0]);residual=X@coef-target;held=np.abs(residual[cut:]);fiterr=np.abs(residual[:cut])
                near=bool(physical and np.median(held)<1e-4 and np.mean(held<1e-4)>=.8)
                rows.append(dict(breath_id=int(bid),R=int(b.R.iloc[0]),C=int(b.C.iloc[0]),family=family,lag=lag,estimator=estimator,eligible_points=len(k),fit_points=cut,held_control_points=len(held),condition=condition,physical=physical,**params,fit_median_abs_control_residual=float(np.median(fiterr)),held_median_abs_control_residual=float(np.median(held)),held_mean_abs_control_residual=float(np.mean(held)),held_control_inlier_fraction=float(np.mean(held<1e-4)),near_exact=near,coefficients=json.dumps(coef.tolist())))
records=pd.DataFrame(rows);records.to_csv(O/'training_only_equation_fits.csv',index=False);summary=[]
for keys,g in records.groupby(['family','lag','estimator'],sort=True):
    summary.append(dict(family=keys[0],lag=int(keys[1]),estimator=keys[2],eligible_breaths=len(g),physical_count=int(g.physical.sum()),median_held_control_residual=float(g.held_median_abs_control_residual.median()),near_exact_count=int(g.near_exact.sum()),near_exact_fraction_of_576=float(g.near_exact.sum()/576)))
summary=sorted(summary,key=lambda r:(-r['near_exact_count'],r['family'],r['lag'],r['estimator']))
report=dict(status='complete_training_only_diagnostic',protocol_sha256=sha(planpath),source_document_sha256=plan['source_document_sha256'],sample_training_breaths=576,same_training_sample_as_V015=True,selected_A_fit_breath_ids=sample_ids.tolist(),no_A_validation_pressure_parsed=True,no_test_data_reads=True,raw_float64=True,diagnostic_units='valve control, NOT pressure MAE',families=summary,gate_passed=bool(any(q['near_exact_fraction_of_576']>=.1 for q in summary)),pressure_predictor_created=False,recovery_coverage_estimated=False,submission_created=False,runtime_seconds=time.monotonic()-started,completed_utc=datetime.now(timezone.utc).isoformat(),interpretation='Conditional control-equation consistency using known pressure on training breaths; not unseen-pressure accuracy. Failure applies only to these reference/decay families, sampling and clipping assumptions.')
(O/'summary.json').write_text(json.dumps(report,indent=2,allow_nan=False)+'\n');print(pd.DataFrame(summary).to_string(index=False));print('Training-only gate:',report['gate_passed'],'CPU seconds:',round(report['runtime_seconds'],2));print('No pressure MAE, input-only reconstruction or submission was produced.')


                         family  lag              estimator  eligible_breaths  physical_count  median_held_control_residual  near_exact_count  near_exact_fraction_of_576
   decay_sum_constant_reference    0 training_only_RANSAC64               466             332                      0.002859                31                    0.053819
   decay_sum_constant_reference    0 ordinary_least_squares               466             266                      0.028399                27                    0.046875
decay_sum_linear_step_reference    0 training_only_RANSAC64               466             310                      0.005713                22                    0.038194
decay_sum_linear_step_reference    0 ordinary_least_squares               466             244                      0.016792                19                    0.032986
    decay_dt_constant_reference    0 ordinary_least_squares               466               0                      0.005029                 0         